# Домашнее задание блока 3. Лекция 5

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IlyaChichkanov/Reinforcement-learning/blob/main/homeworks/hw3/homework.ipynb)

Стратегия с параметрами, REINFORCE и вычитание среднего. В части 1 агент не различает клеток коридора: лучшую стратегию вы посчитаете точно и проверите, что REINFORCE к ней приходит. В части 2 стратегией будет нейросеть, а средой — Acrobot. Код семинара 5 и раздела 5 лекции можно брать за основу.

| Часть | Что проверяем | Баллы |
|---|---|---|
| 1. Коридор с серыми клетками | лучшая стратегия случайна только там, где агент не различает клеток; REINFORCE её находит | 50 |
| 2. Acrobot: стратегия-нейросеть | все награды −1: без вычитания среднего обучение медленнее, а разброс между запусками больше | 50 |

* Ячейки с `assert` проверяют себя сами. Графики и ответы (в markdown вместо `TODO`) проверяет преподаватель.
* Ноутбук считается около 3 минут, на задание уйдёт 2–3 часа.
* Квиз по лекции 5 пройдите в зачётном режиме и положите код результата в `quiz/week05.txt`.
* **Дедлайн:** объявит преподаватель.

Первые две ячейки служебные: запустите их и идите дальше.

In [ ]:
# Если ноутбук открыт в Google Colab: ставим недостающие пакеты. Локально (после uv sync) ячейка ничего не делает.
import importlib.util, subprocess, sys
for package, pip_name in (("gymnasium", "gymnasium[classic-control]"), ("torch", "torch")):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

In [ ]:
# Служебное: модуль с коридором и картинками. Читать не нужно, сигнатуры — в его docstring.
import pathlib, sys, urllib.request

name, folder = "hw3_plots", "homeworks/hw3"
local = next((p for p in ["", f"../../{folder}", f"../../../{folder}", folder] if (pathlib.Path(p) / f"{name}.py").exists()), None)
if local is None:                                                  # в Colab скачиваем из репозитория курса
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/IlyaChichkanov/Reinforcement-learning/main/{folder}/{name}.py", f"{name}.py")
    local = ""
sys.path.insert(0, str(pathlib.Path(local).resolve()))

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
from IPython.display import display
from hw3_plots import (LEFT, RIGHT, OBS_NAMES, GreyCorridor, draw_grey_corridor, plot_value_curves, plot_prob_runs,
                       draw_acrobot, plot_lengths, animate_acrobot)

## Часть 1. Коридор с серыми клетками (50 баллов)

Коридор из раздела 2 лекции, только длиннее и с выходом посередине:

* клетки 0…5, клетка 2 — выход; старт в любой из остальных пяти клеток, все равновероятны;
* действия ← (номер 0) и → (номер 1); шаг в стену оставляет агента на месте;
* награда −1 за шаг, $\gamma = 1$; эпизод кончается на выходе или через 200 шагов;
* датчик видит только «стена слева» (клетка 0), «стена справа» (клетка 5) и «серая клетка»: клетки 1, 3 и 4 для него одинаковы.

Стратегия — три вероятности →: $p_L$, $p_G$ и $p_R$. В серых клетках правильные действия разные: из клетки 1 к выходу ведёт →, из клеток 3 и 4 — ←.

In [ ]:
draw_grey_corridor(title="коридор: сверху номера клеток, снизу — что видит датчик")
plt.show()

env = GreyCorridor()
obs, info = env.reset(seed=0)
print(f"старт в клетке {info['cell']}, датчик видит: {OBS_NAMES[obs]}")
obs, r, terminated, truncated, info = env.step(LEFT)
print(f"после ←: клетка {info['cell']}, награда {r}, выход: {terminated}")

### 1.1 Точная ценность старта (20 баллов)

Ценность клетки подчиняется уравнению Беллмана для стратегии (лекция 2): шаг стоит −1, дальше с вероятностью $p$ агент идёт →, с вероятностью $1 - p$ идёт ←. Ценность выхода $V(2) = 0$. Пять клеток дают пять уравнений:

$$
\begin{aligned}
V(0) &= -1 + p_L\, V(1) + (1 - p_L)\, V(0)\\
V(1) &= -1 + p_G \cdot 0 + (1 - p_G)\, V(0)\\
V(3) &= -1 + p_G\, V(4) + (1 - p_G) \cdot 0\\
V(4) &= -1 + p_G\, V(5) + (1 - p_G)\, V(3)\\
V(5) &= -1 + p_R\, V(5) + (1 - p_R)\, V(4)
\end{aligned}
$$

В клетке 0 шаг ← упирается в стену, поэтому справа стоит $V(0)$; в клетке 5 так же с →.

Перенесите все $V$ в левую часть, и получится система $A\,\vec V = -\vec 1$. Например, первое уравнение превращается в $p_L V(0) - p_L V(1) = -1$, то есть в строку $[p_L, -p_L, 0, 0, 0]$. Ценность старта $J$ — среднее пяти $V$.

**Проверка руками.** При $p_L = 1$, $p_G = 0.5$, $p_R = 0$ получается $V = (-4, -3, -5, -8, -9)$ и $J = -5.8$.

1. **TODO 1.1 (1).** Напишите `start_value(pL, pG, pR)`: матрица $A$, решение `np.linalg.solve`, среднее.
2. **TODO 1.1 (2).** Постройте $J$ против $p_G$ для двух вариантов в клетках 0 и 5: (а) $p_L = 1$, $p_R = 0$, то есть к выходу без случайности; (б) $p_L = p_R = 0.5$. Лучшее $p_G$ для (а) запишите в `BEST_P`, его $J$ — в `BEST_J`.

In [ ]:
def start_value(pL, pG, pR):
    """Ценность старта J — среднее V(0), V(1), V(3), V(4), V(5) — для вероятностей → (pL, pG, pR)."""
    # строки — клетки 0, 1, 3, 4, 5; столбцы — V(0), V(1), V(3), V(4), V(5)
    # клетка 0: p_L·V(0) − p_L·V(1) = −1  →  строка [pL, -pL, 0, 0, 0]
    # TODO 1.1 (1): матрица A из пяти строк, V = np.linalg.solve(A, -np.ones(5)), вернуть среднее V
    raise NotImplementedError

In [ ]:
# Самопроверка 1.1 (1) — пример из текста
J_check = start_value(1.0, 0.5, 0.0)
assert np.isclose(J_check, -5.8), f"при p = (1, 0.5, 0) ценность старта −5.8 (V = −4, −3, −5, −8, −9), у вас {J_check:.3f}"
assert start_value(1.0, 0.4, 0.0) > start_value(1.0, 0.9, 0.0), "чаще → в серых клетках — дольше бродить справа от выхода"
print("OK: J(1, 0.5, 0) =", round(J_check, 3))

In [ ]:
ps = np.linspace(0.02, 0.98, 97)
# TODO 1.1 (2): ценность старта против p_G для двух стратегий в клетках 0 и 5; лучшее p_G для (а) — в BEST_P и BEST_J;
#               картинка: plot_value_curves(ps, {подпись: значения, ...}, best=(BEST_P, BEST_J))
BEST_P, BEST_J = None, None

In [ ]:
# Самопроверка 1.1 (2)
assert BEST_P is not None and 0.28 < BEST_P < 0.38, f"лучшее p_G — около 1/3, у вас {BEST_P}"
assert np.isclose(BEST_J, start_value(1.0, BEST_P, 0.0)), "BEST_J — ценность старта при BEST_P для стратегии (а)"
print(f"OK: лучшее p_G = {BEST_P:.2f}, ценность {BEST_J:.2f}")

**Вопросы к 1.1.** (а) Что происходит с $J$ при $p_G \to 0$ и при $p_G \to 1$? Проследите, куда ходит агент. (б) Почему в клетках 0 и 5 случайность только вредит, а в серых клетках без неё нельзя?

_Ваш ответ:_ TODO

### 1.2 REINFORCE с тремя предпочтениями (30 баллов)

Стратегия — три предпочтения $\vec\theta = (\theta_L, \theta_G, \theta_R)$, вероятность → при наблюдении $o$ равна $p_o = e^{\theta_o} / (1 + e^{\theta_o})$ (раздел 3 лекции). Производная $\log \pi(a \mid o)$ по $\theta_o$ равна $[a = {\rightarrow}] - p_o$, по двум другим предпочтениям — нулю.

Шаг REINFORCE с вычитанием среднего (разделы 4–5 лекции) при $\gamma = 1$:

$$
\vec\theta \;\leftarrow\; \vec\theta + \alpha \sum_{t} (G_t - b)\, \nabla_{\vec\theta} \log \pi(a_t \mid o_t) ,
$$

где $G_t$ — сумма наград с шага $t$ до конца, то есть минус число оставшихся шагов, а $b$ — средний $G_t$ всех прошлых шагов. Шаг $t$ добавляет $(G_t - b)\,([a_t = {\rightarrow}] - p_{o_t})$ к компоненте $o_t$.

**Проверка руками.** Из клетки 0: → (датчик «стена слева», $o = 0$), затем → из клетки 1 («серая», $o = 1$), выход. $G_0 = -2$, $G_1 = -1$, при $\vec\theta = 0$ все $p = 0.5$. При $b = 0$ оценка равна $(-1, -0.5, 0)$, при $b = -1.5$ — $(-0.25, 0.25, 0)$.

1. **TODO 1.2 (1).** Напишите `grad_estimate(theta, observations, actions, rewards, b)`.
2. **TODO 1.2 (2).** Допишите в `reinforce` строку шага, обучите 10 запусков по 3000 эпизодов с $\alpha = 0.01$ и нарисуйте $p_G$ по эпизодам вместе с точным `BEST_P`.

In [ ]:
def p_right(theta):
    """Вероятность → при предпочтении θ (предпочтение ← равно 0); θ может быть массивом."""
    return np.exp(theta) / (1 + np.exp(theta))

def grad_estimate(theta, observations, actions, rewards, b=0.0):
    """Σ_t (G_t − b) ∇ log π(a_t | o_t) — вектор из трёх чисел."""
    # TODO 1.2 (1): G_t — суммы наград с конца; к компоненте observations[t] прибавить (G_t − b) · ([a_t = →] − p)
    raise NotImplementedError

In [ ]:
# Самопроверка 1.2 (1) — эпизод из текста
toy = dict(observations=[0, 1], actions=[RIGHT, RIGHT], rewards=np.array([-1.0, -1.0]))
g0 = grad_estimate(np.zeros(3), **toy)
assert np.allclose(g0, [-1, -0.5, 0]), f"при b = 0 оценка (−1, −0.5, 0), у вас {g0}"
g1 = grad_estimate(np.zeros(3), **toy, b=-1.5)
assert np.allclose(g1, [-0.25, 0.25, 0]), f"при b = −1.5 оценка (−0.25, 0.25, 0), у вас {g1}"
print("OK:", g0, g1)

In [ ]:
def play(env, theta, rng):
    """Один эпизод стратегией с предпочтениями theta; возвращает наблюдения, действия и награды."""
    obs, _ = env.reset(seed=int(rng.integers(1_000_000)))
    observations, actions, rewards = [], [], []
    while True:
        a = int(rng.random() < p_right(theta[obs]))
        observations.append(obs); actions.append(a)
        obs, r, terminated, truncated, _ = env.step(a)
        rewards.append(r)
        if terminated or truncated:
            return observations, actions, np.array(rewards)

def reinforce(alpha=0.01, n_episodes=3000, seed=0):
    """REINFORCE с вычитанием среднего; возвращает вероятности → (p_L, p_G, p_R) после каждого эпизода."""
    rng = np.random.default_rng(seed)
    env, theta = GreyCorridor(), np.zeros(3)
    b, n, history = 0.0, 0, []
    for _ in range(n_episodes):
        observations, actions, rewards = play(env, theta, rng)
        # TODO 1.2 (2): одна строка — шаг θ ← θ + α · grad_estimate(...) со средним b
        raise NotImplementedError
        for g in np.cumsum(rewards[::-1])[::-1]:       # b — средний G_t всех прошлых шагов
            n += 1; b += (g - b) / n
        history.append(p_right(theta))
    return np.array(history)

# TODO 1.2 (2): 10 запусков (seed 0…9), массив runs формы (10, 3000, 3); картинка plot_prob_runs(runs[:, :, 1], target=BEST_P)
runs = None

In [ ]:
# Самопроверка 1.2 (2)
assert runs is not None and runs.shape == (10, 3000, 3), "runs — массив (10 запусков, 3000 эпизодов, 3 вероятности)"
final = runs[:, -1]
assert abs(np.median(final[:, 1]) - BEST_P) < 0.08, \
    f"медиана p_G в конце должна быть близко к точному {BEST_P:.2f}, у вас {np.median(final[:, 1]):.2f}"
assert np.sum((final[:, 0] > 0.9) & (final[:, 2] < 0.15)) >= 8, \
    "в клетках 0 и 5 большинство запусков должно прийти к стратегии без случайности: p_L > 0.9, p_R < 0.15"
print("OK: медиана p_G =", round(np.median(final[:, 1]), 2))

**Вопросы к 1.2.** (а) Почему запуски разбросаны вокруг `BEST_P`? Посмотрите на кривую (а) из 1.1 около вершины. (б) Q-learning по таблице выбрал бы в серых клетках одно действие. Чем это кончится?

_Ваш ответ:_ TODO

## Часть 2. Acrobot: стратегия-нейросеть (50 баллов)

Раздел 5 лекции на новой среде `Acrobot-v1`:

* два звена на шарнирах, верхний шарнир закреплён, в среднем стоит мотор;
* состояние — 6 чисел ($\cos$ и $\sin$ двух углов, две угловые скорости), действия — момент мотора $-1$, $0$, $+1$ (номера 0, 1, 2);
* награда −1 за шаг; эпизод кончается, когда конец нижнего звена поднимется выше пунктира, или через 500 шагов.

Return — минус число шагов, поэтому мерка всей части — **шагов до подъёма**: чем меньше, тем лучше, 500 значит «не поднял». Стратегия — сеть 6 → 32 → 3, вероятности действий — softmax её выхода. Функции `play_net` (эпизод стратегией-сетью, как на семинаре 5) и `returns_to_go` ($G_t$ с конца эпизода) даны готовыми.

In [ ]:
acro = gym.make("Acrobot-v1")
s0, _ = acro.reset(seed=0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
draw_acrobot(s0, ax=axes[0], title="старт:\nзвенья висят вниз")
draw_acrobot([np.cos(2.6), np.sin(2.6), np.cos(-0.9), np.sin(-0.9), 0, 0], ax=axes[1], title="конец звена выше пунктира:\nэпизод окончен")
plt.show()

def make_policy(seed):
    """Стратегия-нейросеть 6 → 32 → 3 с заданным начальным seed."""
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Linear(6, 32), torch.nn.Tanh(), torch.nn.Linear(32, 3))

def returns_to_go(rewards, gamma):
    """G_t = r_t + γ G_{t+1} для каждого шага эпизода, считаем с конца."""
    G, acc = np.zeros(len(rewards)), 0.0
    for t in reversed(range(len(rewards))):
        acc = rewards[t] + gamma * acc
        G[t] = acc
    return G

def play_net(env, net, rng, seed=None):
    """Один эпизод стратегией-сетью; возвращает списки состояний, действий и наград."""
    s, _ = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    while True:
        with torch.no_grad():
            probs = torch.softmax(net(torch.as_tensor(s, dtype=torch.float32)), dim=-1).numpy().astype(float)
        a = int(rng.choice(3, p=probs / probs.sum()))      # деление убирает ошибку округления суммы вероятностей
        s2, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r); s = s2
        if terminated or truncated:
            return states, actions, rewards

print("G_t для наград (−1, −1, −1) при γ = 0.5:", returns_to_go([-1.0, -1.0, -1.0], 0.5))

### 2.1 Функция потерь (15 баллов)

**TODO 2.1.** Напишите функцию потерь раздела 5 лекции `reinforce_loss(net, states, actions, rewards, gamma, b)`:

$$
\text{loss}(\theta) \;=\; -\sum_{t=0}^{T-1} \gamma^t\, (G_t - b)\, \log \pi_\theta(a_t \mid s_t) .
$$

`log_softmax` выхода сети даёт $\log \pi$ всех действий; из каждой строки берите столбец сыгранного действия.

**Проверка руками.** У сети с нулевыми весами $\log \pi = -\log 3$. Два шага с наградами $-1, -1$ и $\gamma = 0.5$: $G = (-1.5, -1)$, веса $\gamma^t G_t = (-1.5, -0.5)$, $\text{loss} = -2 \log 3 \approx -2.197$. При $b = -1$ веса $(-0.5, 0)$ и $\text{loss} = -0.5 \log 3 \approx -0.549$.

In [ ]:
def reinforce_loss(net, states, actions, rewards, gamma=0.99, b=0.0):
    """loss = −Σ_t γ^t (G_t − b) log π(a_t | s_t); тензор из одного числа."""
    # TODO 2.1: G_t — returns_to_go(rewards, gamma); log_softmax выхода сети для всех состояний;
    #           столбцы сыгранных действий; loss
    raise NotImplementedError

In [ ]:
# Самопроверка 2.1
zero = torch.nn.Sequential(torch.nn.Linear(6, 3)); torch.nn.init.zeros_(zero[0].weight); torch.nn.init.zeros_(zero[0].bias)
toy = dict(states=[np.zeros(6), np.zeros(6)], actions=[0, 2], rewards=[-1.0, -1.0], gamma=0.5)
assert np.isclose(reinforce_loss(zero, **toy).item(), -2 * np.log(3), atol=1e-5), "при b = 0 loss = −2 log 3 ≈ −2.197"
assert np.isclose(reinforce_loss(zero, **toy, b=-1.0).item(), -0.5 * np.log(3), atol=1e-5), "при b = −1 loss = −0.5 log 3 ≈ −0.549"
rng = np.random.default_rng(0)
lengths_random = [len(play_net(acro, make_policy(k), rng, seed=k)[2]) for k in range(10)]
print(f"OK: необученная сеть поднимает конец в среднем за {np.mean(lengths_random):.0f} шагов (лимит 500)")

### 2.2 С вычитанием среднего и без (25 баллов)

Каркас `train` играет эпизоды и после каждого делает шаг Adam по `reinforce_loss` (`lr` $= 0.005$, $\gamma = 0.99$, 300 эпизодов).

1. **TODO 2.2 (1).** Допишите в `train` строку `loss`: с $b$ при `baseline=True`, с нулём иначе.
2. **TODO 2.2 (2).** Обучите по 5 запусков (seed 0…4) без вычитания среднего и с ним и нарисуйте `plot_lengths` для обоих вариантов на двух панелях. Мерка запуска — средняя длина его последних 30 эпизодов. Счёт займёт около 2 минут: 10 запусков по 300 эпизодов.

In [ ]:
def train(seed, baseline, n_episodes=300, lr=0.005, gamma=0.99):
    """REINFORCE для стратегии-сети; возвращает длины эпизодов и сеть."""
    rng, net = np.random.default_rng(seed), make_policy(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    env = gym.make("Acrobot-v1")
    b, n, lengths = 0.0, 0, []
    for episode in range(n_episodes):
        states, actions, rewards = play_net(env, net, rng, seed=seed * 10_000 + episode)
        # TODO 2.2 (1): одна строка — loss = reinforce_loss(...) с b при baseline=True и с 0.0 иначе
        raise NotImplementedError
        opt.zero_grad(); loss.backward(); opt.step()
        for g in returns_to_go(rewards, gamma):        # b — средний G_t всех прошлых шагов
            n += 1; b += (g - b) / n
        lengths.append(len(rewards))
    return np.array(lengths), net

# TODO 2.2 (2): по 5 запусков без вычитания и с ним — массивы длин lengths_plain, lengths_base формы (5, 300),
#               сети запусков с вычитанием — в список nets_base; две панели plot_lengths
lengths_plain, lengths_base, nets_base = None, None, None

In [ ]:
# Самопроверка 2.2
assert lengths_base is not None and lengths_base.shape == (5, 300) and lengths_plain.shape == (5, 300), \
    "lengths_plain и lengths_base — массивы (5 запусков, 300 эпизодов)"
last_base, last_plain = lengths_base[:, -30:].mean(axis=1), lengths_plain[:, -30:].mean(axis=1)
assert np.sum(last_base < 200) >= 2, f"с вычитанием среднего хотя бы два запуска из пяти должны поднимать конец быстрее 200 шагов: {last_base.round(0)}"
print(f"OK: медиана последних 30 эпизодов — без вычитания {np.median(last_plain):.0f}, с вычитанием {np.median(last_base):.0f} шагов")

**Вопросы к 2.2.** (а) Какой вариант учится быстрее и у какого меньше разброс между запусками? Какой знак у всех $G_t$ в Acrobot и что без вычитания среднего шаг делает с вероятностью каждого сыгранного действия? (б) Если запуск так и остался у 500 шагов, поднял ли он конец хоть раз (`lengths_base[k]`)? Чему тогда может научиться шаг REINFORCE?

_Ваш ответ:_ TODO

### 2.3 Как поднимает конец обученная сеть (10 баллов)

**TODO 2.3.** Сыграйте эпизод сетью лучшего запуска с вычитанием среднего без случайности: в каждом состоянии берите действие с наибольшим выходом сети. Состояния соберите в `states_best`, действия — в `actions_best`; `animate_acrobot` сделает из них GIF.

In [ ]:
best = int(np.argmin(lengths_base[:, -30:].mean(axis=1)))
net_best = nets_base[best]
s, _ = acro.reset(seed=123)
states_best, actions_best = [s], []
# TODO 2.3: цикл — действие с наибольшим выходом сети (под torch.no_grad()), шаг среды, запись состояния и действия;
#           выход из цикла при terminated или truncated

print("шагов до подъёма:", len(actions_best))
display(animate_acrobot(states_best, actions_best))

In [ ]:
# Самопроверка 2.3
assert len(actions_best) > 0 and len(states_best) == len(actions_best) + 1, "в states_best на одно состояние больше, чем действий"
assert len(actions_best) < 500, "обученная сеть должна поднять конец звена раньше лимита в 500 шагов"
print("OK")

## Чек-лист перед сдачей

- [ ] 1.1: проверка −5.8, кривые, `BEST_P` и `BEST_J`, ответы
- [ ] 1.2: `grad_estimate`, 10 запусков, картинка $p_G$, ответы
- [ ] 2.1: `reinforce_loss` проходит проверку
- [ ] 2.2: по 5 запусков без вычитания среднего и с ним, ответы
- [ ] 2.3: эпизод обученной сети и GIF
- [ ] Код квиза по лекции 5 лежит в `quiz/week05.txt`
- [ ] «Restart & Run All» перед сдачей